In [1]:
import io
import boto3
import numpy as np
import pandas as pd

s3 = boto3.client("s3", region_name="us-east-2")
BUCKET = "demand-forecast-ogum"

obj = s3.get_object(Bucket=BUCKET, Key="raw/vendas_historicas.csv")
df = pd.read_csv(io.BytesIO(obj["Body"].read()), parse_dates=["data"])
print(f"Linhas brutas: {len(df):,}")
df.head()

Linhas brutas: 36,575


,data,produto,regiao,quantidade_vendida,preco_unitario,receita
0,2022-01-01,Produto_A,Sudeste,99.0,84.41,8356.59
1,2022-01-02,Produto_A,Sudeste,111.0,84.41,9369.51
2,2022-01-03,Produto_A,Sudeste,86.0,83.23,7157.78
3,2022-01-04,Produto_A,Sudeste,78.0,91.53,7139.34
4,2022-01-05,Produto_A,Sudeste,65.0,84.64,5501.60


In [2]:
print("Nulos por coluna:")
print(df.isna().sum())
print("\nLinhas duplicadas:", df.duplicated().sum())
print("\nResumo de quantidade_vendida:")
print(df["quantidade_vendida"].describe().round(1))

Nulos por coluna:
data                    0
produto                 0
regiao                  0
quantidade_vendida    732
preco_unitario          0
receita                 0
dtype: int64

Linhas duplicadas: 50

Resumo de quantidade_vendida:
count    35843.0
mean        70.2
std         54.7
min         17.0
25%         48.0
50%         65.0
75%         85.0
max       3520.0
Name: quantidade_vendida, dtype: float64


In [4]:
linhas_brutas = len(df)

#  Remover linhas duplicadas
df = df.drop_duplicates().reset_index(drop=True)

#  Preencher nulos com a mediana do mesmo produto + região + dia da semana
df["dia_semana"] = df["data"].dt.dayofweek
mediana = df.groupby(["produto", "regiao", "dia_semana"])["quantidade_vendida"].transform("median")
df["quantidade_vendida"] = df["quantidade_vendida"].fillna(mediana)

#  Vendas absurdas: limitar a um teto calculado por produto
def teto(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    return q3 + 3 * (q3 - q1)

limite = df.groupby("produto")["quantidade_vendida"].transform(teto)
n_outliers = int((df["quantidade_vendida"] > limite).sum())
df["quantidade_vendida"] = np.where(
    df["quantidade_vendida"] > limite, limite, df["quantidade_vendida"]
).round().astype(int)

#  Recalcular a receita, porque a quantidade mudou
df["receita"] = (df["quantidade_vendida"] * df["preco_unitario"]).round(2)
df = df.drop(columns="dia_semana")

In [5]:
print(f"Linhas: {linhas_brutas:,} -> {len(df):,}")
print("Nulos:", int(df.isna().sum().sum()), "| Duplicatas:", int(df.duplicated().sum()), "| Vendas limitadas ao teto:", n_outliers)
df["quantidade_vendida"].describe().round(1)

Linhas: 36,525 -> 36,525
Nulos: 0 | Duplicatas: 0 | Vendas limitadas ao teto: 0


count    36525.0
mean        69.0
std         27.4
min         17.0
25%         48.0
50%         65.0
75%         84.0
max        232.0
Name: quantidade_vendida, dtype: float64

In [6]:
buffer = io.StringIO()
df.to_csv(buffer, index=False)
s3.put_object(
    Bucket=BUCKET,
    Key="processed/vendas_historicas_limpo.csv",
    Body=buffer.getvalue().encode("utf-8"),
)
print("Salvo em s3://demand-forecast-ogum/processed/vendas_historicas_limpo.csv")

Salvo em s3://demand-forecast-ogum/processed/vendas_historicas_limpo.csv
